# HW1 — analytical performance model of SmallCNN

Thin runner on top of the repo files (`models.py`, `equations.py`, `measure.py`,
`calibrate.py`, `make_figures.py`) — no code is duplicated here.

**Runtime → Change runtime type → T4 GPU.** Run cells top to bottom.
Full grid on T4: **~40–70 min** (energy bursts dominate).
Smoke steps take ~2 min.

**Memory budget:** `measure.py` runs the process under
`torch.cuda.set_per_process_memory_fraction` with a **2 GB budget**
(`--mem-limit-gb`, default `2`). This net fits even a 16 GB T4 uncapped
(max ≈ 4.6 GB), so without the budget the assignment's "catch OOM and compare
with the Memory equation" step would be vacuous. With 2 GB the largest-S/largest-B
corner (exactly 11 configs: (384,256), (448,256), (496|512)×large B) OOMs, and we compare that boundary with the equation.

In [ ]:
!nvidia-smi
!pip -q install nvidia-ml-py

**Get the code:** either clone your repo (paste the URL into `REPO`), or leave it
empty and upload the five files when asked.

In [ ]:
import os
REPO = ""  # e.g. "https://github.com/<user>/<repo>.git"; leave empty if you upload files
if REPO:
    !git clone -q {REPO} /content/repo
    %cd /content/repo/hw1
else:
    os.makedirs('/content/hw1', exist_ok=True)
    %cd /content/hw1
    need = ['models.py','equations.py','measure.py','calibrate.py','make_figures.py']
    if not all(os.path.exists(f) for f in need):
        from google.colab import files
        print("upload:", ", ".join(need))
        files.upload()
!ls

Sanity check: the network and the four closed forms (formulas and table must
agree with the model — `equations.py` asserts the weight count).

In [ ]:
!python models.py
!python equations.py | head -20

Smoke test (small grid, writes to `results_quick/`, ~2 min — does not touch
the real results).

In [ ]:
!python measure.py --quick --out results_quick/measurements.csv
!tail -3 results_quick/measurements.csv | cut -d, -f1-7

**Full grid — 132 configurations** (11 S × 12 B, seed=42; 69 unseen validation
points). `--resume` makes the cell idempotent: already-measured (S,B) rows are
skipped and the run appends to the existing CSV — **just re-run this cell after a
Colab disconnect**. To start clean, `!rm results/measurements.csv`.

In [ ]:
!python measure.py --resume

Calibrate θ on the base grid only (Nelder-Mead in log-space for latency,
non-negative least squares for energy), report MAPE on calibration **and** unseen
validation points, write `results/theta.json`.

In [ ]:
!python calibrate.py

Figures: measured points together with predicted curves/surfaces.

In [ ]:
!python make_figures.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(f, width=900))

In [ ]:
!zip -qr results.zip results results_quick
from google.colab import files
files.download('results.zip')

# alternative: keep it in Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r results /content/drive/MyDrive/hw1_results

## What to expect on the T4 (and what to update afterwards)

* **Latency MAPE** ~10–25 % on unseen points; flat `18·t_launch` plateau at small
  S·B (launch-bound), a ∝B·S² slope in the middle (memory-bound). Unlike our H100
  run, the T4 should reach **compute-bound** at large S·B: the net needs ~0.021
  bytes per FLOP (380 B / 17714 FLOP) while T4's roofline crossover
  (BW/P = 300 GB/s ÷ 8.1 TFLOP/s) is ~0.037 B/FLOP — the hardware moves data
  *faster relative to arithmetic* than the net needs, so arithmetic saturates
  first → check `regimes.png`, that's the three-regime story the assignment asks for.
* **Memory**: peak is at MaxPool (PyTorch materializes int64 argmax indices) →
  `4·(1040324 + 17·B·S²)`; measured = formula + cuDNN workspace constant (~10–40 MB)
  ± algorithm switches. Under the 2 GB budget the OOM boundary is
  `B·S² > (2·10⁹/4 − 1 040 324)/17 ≈ 2.9·10⁷` — check `figures/oom_boundary.png`:
  the corner S≥384 with big B must be `OOM` rows; expect one or two borderline
  points where the equation says "fits" but reserved-segment rounding + workspace
  push the run over the limit. (Uncapped, this grid never OOMs even on a 16 GB T4:
  the max point needs ≈ 4.6 GB — worth a sentence in the discussion too.)
* **Energy**: launch-bound configs sit below the whole-GPU sensor floor (zeros are
  expected); FLOPs/Bytes are collinear on this grid, so the e_flop/e_byte split is
  not identifiable — say so.
* Unzip `results.zip` into the repo (`hw1/results/...`), update README tables
  (GPU row, θ, MAPEs) from `results/theta.json` + `metadata.json`, re-run nothing else.
* Still by hand: rewrite `derivations.md` on paper → scan to `hw1_handwritten.pdf`,
  push, submit the GitHub link in the form.

If a `measure.py` run crashed mid-cell for a non-disconnect reason, check the last
printed row and the CSV tail — `status` column shows `ok/OOM/err`.